In [23]:
# Setting up packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from styles.colors import get_colors
from scipy import stats
import re
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import mutual_info_classif

In [5]:
# Load data
data = pd.read_csv("data/processed/filtered_data.csv", index_col=0)

/var/folders/rz/3cfhhdrd2bs65msqx2gtqcph0000gn/T/ipykernel_33162/3614612948.py:2: DtypeWarning: Columns (0: ards_notmild, 1: age) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/processed/filtered_data.csv", index_col=0)


# Load simulated data instead

In [26]:
from src.core import simulate_data

# Train/test + balanced val split

In [6]:
# --- setup ---
test_size = 0.05
val_frac_of_remaining = 0.20
random_state = 42

X = data.drop(columns=["ards"]).copy()
y = data["ards"].astype(int).copy()

# skapa unikt rad-id
row_id = np.arange(len(data))

# 5% test (stratified)
X_rem, X_test, y_rem, y_test, id_rem, id_test = train_test_split(
    X, y, row_id,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

rem_df = X_rem.copy()
rem_df["ards"] = y_rem
rem_df["row_id"] = id_rem

df_pos = rem_df[rem_df["ards"] == 1]
df_neg = rem_df[rem_df["ards"] == 0]

n_val_total = int(round(len(rem_df) * val_frac_of_remaining))
n_each = min(n_val_total // 2, len(df_pos), len(df_neg))

val_pos = df_pos.sample(n=n_each, random_state=random_state)
val_neg = df_neg.sample(n=n_each, random_state=random_state)

val_df = pd.concat([val_pos, val_neg]).sample(frac=1, random_state=random_state)

# split via row_id (inte index)
val_ids = set(val_df["row_id"].values)
train_df = rem_df[~rem_df["row_id"].isin(val_ids)]

X_val = val_df.drop(columns=["ards", "row_id"])
y_val = val_df["ards"]

X_train = train_df.drop(columns=["ards", "row_id"])
y_train = train_df["ards"]

# --- prints ---
def counts(y_):
    return len(y_), int(y_.sum()), int((1 - y_).sum())

for name, ys in [("train", y_train), ("val", y_val), ("test", y_test)]:
    n, n_pos, n_neg = counts(ys)
    print(f"{name:5s}: {n:4d} patients | ARDS: {n_pos:4d} | non-ARDS: {n_neg:4d}")

print("\nFractions:")
print(f"test: {len(y_test)/len(y):.3f}")
print(f"val : {len(y_val)/len(y):.3f}")
print(f"train: {len(y_train)/len(y):.3f}")

train:  311 patients | ARDS:   24 | non-ARDS:  287
val  :   76 patients | ARDS:   38 | non-ARDS:   38
test :   21 patients | ARDS:    3 | non-ARDS:   18

Fractions:
test: 0.051
val : 0.186
train: 0.762


# Univariate Mutual Information

One practical concern: MI estimation from continuous data requires binning or density estimation (e.g. KDE or k-NN based estimators like the Kraskov estimator), and ARDS is quite rare in your cohort (65/409), so the estimates for weak associations will be noisy.

In [20]:
# Top proteins with MI score

# Setup
protein_cols = [c for c in X_train.columns if re.match(r"^seq", str(c))]
X_prot = X_train[protein_cols]
y = y_train.astype(int)

# Beräkna MI för varje protein till label
mi_scores = mutual_info_classif(
    X_prot,
    y,
    discrete_features=False,   # proteiner är kontinuerliga
    n_neighbors=3,
    random_state=42
)

# Samla i dataframe
mi_results = pd.DataFrame({
    "Protein": protein_cols,
    "MI": mi_scores
}).sort_values("MI", ascending=False).reset_index(drop=True)

print(mi_results.head(15))

          Protein        MI
0    seq.24723.58  0.075984
1   seq.12818.159  0.068495
2    seq.18273.14  0.062019
3   seq.31608.334  0.060637
4    seq.23568.41  0.060634
5    seq.10606.34  0.060544
6    seq.21935.16  0.059234
7     seq.7625.27  0.059070
8   seq.13088.397  0.058926
9   seq.10015.119  0.058418
10   seq.12842.43  0.058374
11    seq.2970.60  0.057006
12     seq.2755.8  0.056339
13  seq.33280.242  0.056150
14   seq.33564.18  0.055964


## Permutation tests for significance 

In [22]:
RANDOM_STATE = 42
N_PERM = 1000          # 1000 är okej start. 5000 om du vill ha stabilare små p.
N_NEIGHBORS = 3
CORRECTION_METHOD = "fdr"   # Benjamini–Hochberg

# --- Settings ---
N_PERM = 1000
RANDOM_STATE = 42
N_NEIGHBORS = 3

rng = np.random.default_rng(RANDOM_STATE)

# --- Samma proteinordning som i mi_results ---
protein_cols_mi = mi_results["Protein"].tolist()

# --- X train numeric + median impute ---
X_prot = X_train[protein_cols_mi].apply(pd.to_numeric, errors="coerce")
X_prot = X_prot.fillna(X_prot.median())

y = y_train.astype(int).values

# --- Observed MI (från din dataframe) ---
mi_obs = mi_results["MI"].values

# --- Permutation loop ---
counts = np.zeros(len(protein_cols_mi), dtype=int)

for b in range(N_PERM):
    y_perm = rng.permutation(y)

    mi_perm = mutual_info_classif(
        X_prot.values,
        y_perm,
        discrete_features=False,
        n_neighbors=N_NEIGHBORS,
        random_state=RANDOM_STATE
    )

    counts += (mi_perm >= mi_obs)

# --- Empiriska p-värden ---
p_perm = (counts + 1) / (N_PERM + 1)

# --- Lägg till i dataframe ---
mi_results["p_perm"] = p_perm

# --- Benjamini–Hochberg FDR ---
_, adj_p, _, _ = multipletests(mi_results["p_perm"], method="fdr_bh")
mi_results["ADJ_P"] = adj_p

# Sortera mest signifikanta först
mi_results = mi_results.sort_values(["ADJ_P", "MI"], ascending=[True, False]).reset_index(drop=True)

mi_results.head(15)

KeyboardInterrupt: 

In [ ]:
significant = mi_results[mi_results["ADJ_P"] < 0.05]
print("Antal signifikanta proteiner:", len(significant))
